# myPy5 Remote Render Server (Colab)

This notebook spins up the render server on a Colab GPU and exposes it via a Cloudflare tunnel.

**Workflow:**
1. Run all cells.
2. Copy the tunnel URL printed in the last cell.
3. On your local machine:
   ```
   REMOTE_RENDER_URL=<paste-url-here> python app.py
   ```
4. Open `http://127.0.0.1:5050` locally and hit **Render** — the heavy work happens on Colab.

In [ ]:
# ── 1. Install Python dependencies ──────────────────────────────
!pip install -q flask numpy matplotlib pillow scipy
print("✓ deps installed")

In [ ]:
# ── 2. Clone your repo (edit the URL if you forked) ─────────────
REPO_URL = "https://github.com/hszhai/myPy5.git"
!git clone --depth 1 {REPO_URL} myPy5
%cd myPy5
print("✓ repo cloned")

In [ ]:
# ── 3. Download cloudflared tunnel binary ───────────────────────
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print("✓ cloudflared ready")

In [ ]:
# ── 4. Start the remote render server in the background ─────────
import subprocess
import time

# Change SCENE_NAME to match what you use locally
env = dict(os.environ, SCENE_NAME="redhead")

server_proc = subprocess.Popen(
    ["python", "remote_server.py"],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
time.sleep(3)

print("✓ remote_server.py started on port 5050")

In [ ]:
# ── 5. Launch the Cloudflare tunnel ─────────────────────────────
tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:5050"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
time.sleep(8)
print("✓ tunnel process started")

In [ ]:
# ── 6. Extract and print the public tunnel URL ──────────────────
import re

url = None
for _ in range(30):
    line = tunnel_proc.stdout.readline()
    if not line:
        break
    m = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break
    time.sleep(0.5)

if url:
    print("\n" + "═"*60)
    print("  COPY THIS URL INTO YOUR LOCAL TERMINAL:")
    print(f"    REMOTE_RENDER_URL={url}")
    print("  Then run:  REMOTE_RENDER_URL=<url> python app.py")
    print("═"*60 + "\n")
else:
    print("⚠ Could not extract tunnel URL. Check tunnel logs below:")
    !cat /dev/null  # placeholder so the cell doesn't crash


In [ ]:
# ── (Optional) Stream server + tunnel logs ──────────────────────
import sys
from IPython.display import clear_output

def stream_logs(proc, label):
    for line in iter(proc.stdout.readline, ""):
        print(f"[{label}] {line}", end="")
        sys.stdout.flush()

# Run in background threads so the cell stays alive
import threading
threading.Thread(target=stream_logs, args=(server_proc, "SRV"), daemon=True).start()
threading.Thread(target=stream_logs, args=(tunnel_proc, "TUN"), daemon=True).start()

print("Streaming logs... (stop this cell to interrupt)")
while True:
    time.sleep(1)